In [ ]:
# Kinldy ignore this cell if your data is not resided in your local machine.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Data handling
import numpy as np
import pandas as pd
from collections import Counter

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.compose import ColumnTransformer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# Class imbalance reduction techniques
import shap
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

# Model Evaluation and Splitting
import xgboost as xgb
import lightgbm as lgb
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Visualization
import matplotlib.pyplot as plt

In [ ]:
#path to the dataset
file_path = '<Path to the dataset here>'

try:
  df_data = pd.read_csv(file_path + "dat.csv") # Actual file
  df_md = pd.read_csv(file_path + "dat_md.csv") # Medications file
except FileNotFoundError:
  print(f"Error: File not found at {file_path}")

df_data.head()
df_md.head()

In [ ]:
# Appending medications file to the main dataset.
# Converting repeated patient records into unique rows with 'gropuby' clause and performing merge on 'inpatient.number'.
medications_grouped = df_md.groupby('inpatient.number')['Drug_name'].apply(list).reset_index()
merged_data = pd.merge(df_data, medications_grouped, on='inpatient.number', how='left')
merged_data = merged_data.drop_duplicates(subset='inpatient.number')
merged_data.head()

# This is the dataset obtained by merging dat.csv with dat_md.csv
data = merged_data
data.head()

In [ ]:
# List of irrelevant columns in the dataset.
irr_columns = ['Unnamed: 0',
                  'admission.ward',
                  'admission.way',
                  'occupation',
                  'discharge.department',
                  'death.within.28.days',
                  'death.within.3.months',
                  'death.within.6.months',
                  'time.of.death..days.from.admission.',
                  're.admission.time..days.from.admission.',
                  'time.to.emergency.department.within.6.months',
                  'dischargeDay',
                  're.admission.within.3.months',
                  're.admission.within.6.months',
                  'DestinationDischarge']

# Dropping irrelevant columns.
data = data.drop(columns=irr_columns)

In [ ]:
# This is the function that accepts pandas dataframe.
# It calculates the percentages of null values of each column, visualize in a graph and return a dictionary.

def calculate_null_percentage(data):

  null_percentages = {col: data[col].isnull().sum() / data.shape[0] * 100 for col in data.columns}

  # Create a dataframe to display the percentage of null values for each column
  null_values_percentage = pd.DataFrame.from_dict(null_percentages, orient='index', columns=['Null Percentage'])

  # Plot the null percentages as a bar plot
  null_percentages_series = pd.Series(null_percentages).sort_values()
  plt.figure(figsize=(12, 20))
  null_percentages_series.plot(kind='barh', fontsize=8)
  plt.title('Percentage of Null Values per Column')
  plt.xlabel('Columns')
  plt.ylabel('Percentage of Null Values')
  plt.savefig('null_values_percentage.png')
  return [null_percentages, null_percentages_series]


# Calculate the percentage of null values for each column
(null_percentages, null_percentages_series) = calculate_null_percentage(data)

# Dropping the columns containing more than 50% of null values.
data = data.drop(columns=null_percentages_series[null_percentages_series >= 50].index)

In [ ]:
## Performing Imputation

# Iterative imputation for numerical data
iterative_imputer = IterativeImputer()
numerical_cols = data.select_dtypes(include=['number']).columns

# Apply iterative imputation
data[numerical_cols] = iterative_imputer.fit_transform(data[numerical_cols])

# Imputation for categorical columns
categorical_cols = data.select_dtypes(include=['object']).columns

# Exclude 'Drug_Name' from imputation
categorical_cols = [col for col in categorical_cols if col != 'Drug_name']
for col in categorical_cols:
    data.loc[:, col] = data[col].fillna(data[col].mode()[0])

In [ ]:
# Performing Label Encoding
# Defining Size mapping for ordinal data
size_mapping_for_ageCat = {'(21,29]': 1, '(29,39]': 2, '(39,49]': 3, '(49,59]': 4, '(59,69]': 5, '(69,79]': 6, '(79,89]': 7, '(89,110]': 8}
size_mapping_for_NYHA = {'I': 1, 'II': 2, 'III': 3, 'IV':4}
size_mapping_for_killip = {'I': 1, 'II': 2, 'III': 3, 'IV':4}
size_mapping_for_consciouness = {'Clear': 1, 'ResponsiveToSound': 2, 'ResponsiveToPain': 3, 'Nonresponsive': 4}

# Dictionary to map each column to its respective mapping
mapping_dict = {
    'ageCat': size_mapping_for_ageCat,
    'NYHA.cardiac.function.classification': size_mapping_for_NYHA,
    'Killip.grade': size_mapping_for_killip,
    'consciousness': size_mapping_for_consciouness
}

# Performing Label Encoding
for col in mapping_dict.keys():
    data[col] = data[col].map(mapping_dict[col])

In [ ]:
# Performing OneHot encoding
# List of columns for non-ordinal data
one_hot_columns = ['gender','type.of.heart.failure','type.II.respiratory.failure','oxygen.inhalation','outcome.during.hospitalization']

#Performing One-Hot Encoding and dropping first column to avoid dummy variable trap
data = pd.get_dummies(data, columns=one_hot_columns, drop_first=True, dtype=int)

In [ ]:
# Embedding the values of type, List of strings in "Drug_name" column.
# Convert lists to strings safely, handling NaN values
data['Drug_name_str'] = data['Drug_name'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['Drug_name_str'])
data['Drug_name_seq'] = tokenizer.texts_to_sequences(data['Drug_name_str'])

# Padding sequences
max_length = max(data['Drug_name_seq'].apply(len))  # Max length of any sequence
data['Drug_name_seq'] = pad_sequences(data['Drug_name_seq'], maxlen=max_length, padding='post').tolist()

# Drop original columns
data = data.drop(columns=['Drug_name', 'Drug_name_str'])

In [ ]:
# Partitioning of dependent and independent variables.
X = data.drop(columns=['re.admission.within.28.days'])
y = data['re.admission.within.28.days']

In [ ]:
# Peforming Standardization except for one column.
scaler = StandardScaler()
X[X.columns.difference(['Drug_name_seq'])] = scaler.fit_transform(X[X.columns.difference(['Drug_name_seq'])])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Convert list elements in 'Drug_name_seq' into separate columns
max_length = len(X_train['Drug_name_seq'].iloc[0])  # Get the sequence length

# Create separate columns for each element in the sequence
for i in range(max_length):
    X_train[f'Drug_name_seq_{i}'] = X_train['Drug_name_seq'].apply(lambda x: x[i] if len(x) > i else 0)
    X_test[f'Drug_name_seq_{i}'] = X_test['Drug_name_seq'].apply(lambda x: x[i] if len(x) > i else 0)

# Drop the original 'Drug_name_seq' column
X_train = X_train.drop(columns=['Drug_name_seq'])
X_test = X_test.drop(columns=['Drug_name_seq'])

# Encoding y_train
y_train = LabelEncoder().fit_transform(y_train)

In [ ]:
# Visualization of readmissions in the dataset
data = df_data
total_patients = len(data)
readmitted_28_days = data['re.admission.within.28.days'].sum()
readmitted_3_months = data['re.admission.within.3.months'].sum()
readmitted_6_months = data['re.admission.within.6.months'].sum()
never_readmitted = total_patients - data[['re.admission.within.28.days', 're.admission.within.3.months', 're.admission.within.6.months']].any(axis=1).sum()

# Astacked bar chart
fig, ax = plt.subplots(figsize=(6, 9))
categories = ['Never Readmitted', 'Readmitted']
bar_width = 0.6
ax.bar(categories[0], never_readmitted, color='green', label='Never Readmitted')
b1 = ax.bar(categories[1], readmitted_28_days, color='blue', label='Readmitted (28 Days)')
b2 = ax.bar(categories[1], readmitted_3_months, color='orange', bottom=readmitted_28_days, label='Readmitted (3 Months)')
b3 = ax.bar(categories[1], readmitted_6_months, color='red', bottom=readmitted_28_days + readmitted_3_months, label='Readmitted (6 Months)')
ax.set_ylabel('Number of Patients')
ax.set_title(f'Total Patients: {total_patients}\nComparison of Readmitted vs. Never Readmitted Patients')

# Legend
ax.legend(loc='upper left', bbox_to_anchor=(1, 2))

plt.tight_layout()
#plt.show()

In [ ]:
# Defining LBG model for SHAP values
lgb_model = lgb.LGBMClassifier(objective='binary', metric='auc', n_estimators=100, learning_rate=0.1)
lgb_model.fit(X_train, y_train)

In [ ]:
# Visualizing the SHAP Values
explainer = shap.Explainer(lgb_model, X_train)
shap_values = explainer(X_test, check_additivity=False)
shap.summary_plot(shap_values, X_test, max_display=25)

# Feature importance calculation
feature_importance = np.abs(shap_values.values).mean(axis=0)
feature_importance_df = pd.DataFrame({'Feature': X_test.columns, 'SHAP Importance': feature_importance})
feature_importance_df = feature_importance_df.sort_values(by='SHAP Importance', ascending=False)

print(feature_importance_df)

# Categorizing the low importamt features by mean

mean_importance = feature_importance_df["SHAP Importance"].mean()
low_importance_features = feature_importance_df[feature_importance_df["SHAP Importance"] < mean_importance]

print("Low importance features:\n", low_importance_features)

# Truncating the least important features from the dataset
for col in low_importance_features["Feature"]:
  X_train = X_train.drop(columns=col)
  X_test = X_test.drop(columns=col)

In [ ]:
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')),
    ('lgbm', LGBMClassifier(n_estimators=100, learning_rate=0.1, class_weight='balanced')),
    ('svm', SVC(kernel='rbf', C=2.0, gamma='scale', class_weight='balanced', probability=True, random_state=42))
]

meta_model = LogisticRegression(max_iter=1000, class_weight='balanced')

stacked_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)

In [ ]:
stacked_model.fit(X_train, y_train)

In [ ]:
y_pred_stack = stacked_model.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stack))
print("Classification Report:\n", classification_report(y_test, y_pred_stack))

In [ ]:
# Assuming y_test and y_pred_stack are already defined from the previous code
cm = confusion_matrix(y_test, y_pred_stack)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix for Stacked Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
threshold = 0.45 # Tune the threshold here
y_pred_thresh = (y_proba >= threshold).astype(int)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_thresh))
print("Classification Report:\n", classification_report(y_test, y_pred_thresh))